# 10. Обучение трансформера

**Цель:** Реализовать полный цикл обучения: функцию потерь, оптимизатор, планировщик LR, мониторинг, сохранение модели и эксперименты с гиперпараметрами.

---

In [ ]:
import sys, os, logging, math

import torch  # Фреймворк глубокого обучения
import torch.nn as nn  # Слои нейросети
import torch.nn.functional as F  # Функции (softmax, cross_entropy)
import numpy as np  # Численные операции
import matplotlib.pyplot as plt  # Визуализация
from copy import deepcopy  # Глубокое копирование словарей

if torch.cuda.is_available():  # GPU NVIDIA
    device = torch.device("cuda")

elif torch.backends.mps.is_available():  # GPU Apple
    device = torch.device("mps")

else:  # CPU
    device = torch.device("cpu")

## 10.1 Функция потерь с игнорированием padding

CrossEntropyLoss с `ignore_index=PAD` — токены PAD не участвуют в подсчёте loss.

In [ ]:
BOS, EOS, PAD = 0, 1, 2  # Специальные токены
criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # PAD не участвует в loss

## 10.2 Noam Scheduler

**Формула (Vaswani et al.):**
$$\text{lr} = d_{model}^{-0.5} \cdot \min(\text{step}^{-0.5}, \text{step} \cdot \text{warmup}^{-1.5})$$

**Фазы:**
1. Linear warmup: LR растёт от 0 до пика за warmup_steps
2. Decay: LR убывает обратно пропорционально sqrt(step)

In [ ]:

class NoamScheduler:  # Планировщик скорости обучения (Vaswani et al.)
    def __init__(self, optimizer, d_model, warmup_steps=4000, factor=1.0):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self._step = 0
        self._rate = 0
    
    def step(self):  # Обновление LR и шаг оптимизатора
        self._step += 1
        self._rate = self.factor * (self.d_model ** -0.5) * min(self._step ** -0.5, self._step * self.warmup_steps ** -1.5)  # Формула Noam: warmup + decay
        for p in self.optimizer.param_groups:
            p['lr'] = self._rate  # Устанавливаем LR для всех групп
        self.optimizer.step()
    
    def get_rate(self):  # Возвращает текущий LR
        return self._rate

# Визуализация LR schedule
d_model = 32  # Размерность для демонстрации
optimizer_dummy = torch.optim.Adam([torch.tensor(0.)], lr=0)  # Фиктивный оптимизатор
scheduler = NoamScheduler(optimizer_dummy, d_model, warmup_steps=20)  # Warmup = 20 шагов
lrs = []  # Собираем LR на каждом шаге
for _ in range(200):  # Симулируем 200 шагов
    scheduler.step()  # Шаг оптимизатора + обновление LR
    lrs.append(scheduler.get_rate())

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.axvline(20, color='red', linestyle='--', alpha=0.5, label='Warmup end')  # Граница warmup
plt.xlabel('Step')
plt.ylabel('Learning rate')
plt.title(f'Noam Scheduler (d_model={d_model}, warmup=20)')
plt.legend()
plt.grid(True)
plt.show()

## 10.3 Компоненты трансформера (минимум для обучения)

In [ ]:
class MultiHeadAttention(nn.Module):  # Многоголовое внимание
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0  # d_model кратен n_heads
        self.d_k = d_model // n_heads  # Размерность головы
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # Query
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # Key
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # Value
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # Output
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        n_heads = self.W_Q.out_features // self.d_k  # Количество голов
        Q = self.W_Q(Q).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch, heads, seq, seq)
        if mask is not None:  # Применяем маску
            scores = scores.masked_fill(mask == 0, float('-inf'))  # -inf для маскированных позиций
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.W_O.in_features)  # Собираем головы обратно
        return self.W_O(output)  # Выходная проекция

class FeedForward(nn.Module):  # Двухслойная FFN
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # Скрытая размерность (4xd_model)
        self.fc1 = nn.Linear(d_model, d_ff)  # Расширение d_model -> d_ff
        self.fc2 = nn.Linear(d_ff, d_model)  # Сжатие d_ff -> d_model
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))  # Linear -> GELU -> Dropout -> Linear

class EncoderBlock(nn.Module):  # Блок энкодера
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask=None):  # Self-Attention + FFN с Add&Norm
        x = x + self.dropout1(self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask))  # Self-Attention + Add&Norm
        x = x + self.dropout2(self.ffn(self.norm2(x)))  # FFN + Add&Norm
        return x

class Transformer(nn.Module):  # Encoder-only трансформер
    def __init__(self, vocab_size, d_model, n_heads, num_layers, max_len=100, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)  # Эмбеддинги токенов
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
        self.dropout_pe = nn.Dropout(dropout)  # Dropout для PE
        self.layers = nn.ModuleList([EncoderBlock(d_model, n_heads, 4*d_model, dropout) for _ in range(num_layers)])  # Стек энкодеров
        self.output_proj = nn.Linear(d_model, vocab_size)  # Выходная проекция
        for p in self.parameters():  # Инициализация Xavier
            if p.dim() > 1:  # Только весовые матрицы
                nn.init.xavier_uniform_(p)  # Xavier Glorot
    
    def forward(self, x, mask=None):  # Self-Attention + FFN с Add&Norm
        x = self.dropout_pe(self.embedding(x) * math.sqrt(self.d_model) + self.pe[:, :x.size(1), :])  # Эмбеддинги + PE + Dropout
        for layer in self.layers:  # Проход по всем слоям
            x = layer(x, mask)  # Прямой проход через блок
        return self.output_proj(x)  # Выход: логиты для каждого токена

## 10.4 Цикл обучения с мониторингом

Включает: train/val loss, accuracy, early stopping, сохранение модели.

In [ ]:

vocab_size = 16  # Размер словаря

def make_data(num_samples, max_len):  # Генерация данных для копирования
    src, tgt = [], []
    for _ in range(num_samples):
        length = np.random.randint(2, max_len + 1)  # Случайная длина
        seq = np.random.randint(3, vocab_size, size=length).tolist()  # Случайные токены
        src.append([BOS] + seq + [EOS] + [PAD] * (max_len - length))  # [BOS, ..., EOS, PAD...]
        tgt.append([BOS] + seq + [EOS] + [PAD] * (max_len - length))  # Цель: [BOS, ..., EOS, PAD...]
    return torch.tensor(src), torch.tensor(tgt)

train_src, train_tgt = make_data(800, 8)  # 800 обучающих примеров
val_src, val_tgt = make_data(200, 8)  # 200 валидационных

model = Transformer(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=3, max_len=12).to(device)  # Encoder-only трансформер
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, betas=(0.9, 0.98), eps=1e-9)  # AdamW с параметрами из статьи
scheduler = NoamScheduler(optimizer, d_model=32, warmup_steps=50)  # Noam scheduler
criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # PAD не участвует в loss

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:

n_epochs = 60  # Максимум 60 эпох
batch_size = 32  # Размер мини-батча
best_loss = float('inf')  # Лучший validation loss
patience = 10  # Ранняя остановка: 10 эпох без улучшения
wait = 0  # Счётчик эпох без улучшения
best_state = None  # Лучшие веса модели

train_losses, val_losses, lr_history = [], [], []  # Логи обучения

for epoch in range(n_epochs):  # Цикл обучения
    model.train()  # Режим обучения
    train_loss = 0
    perm = torch.randperm(len(train_src))  # Перемешивание данных
    
    for i in range(0, len(train_src), batch_size):  # Мини-батчи
        idx = perm[i:i+batch_size]
        src = train_src[idx].to(device)  # Исходные токены
        tgt = train_tgt[idx].to(device)  # Целевые токены
        
        output = model(src)  # Прямой проход
        loss = criterion(output.reshape(-1, vocab_size), tgt.reshape(-1))  # Loss между предсказанием и true
        
        optimizer.zero_grad()  # Обнуление градиентов
        loss.backward()  # Обратное распространение
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Клиппинг градиентов
        scheduler.step()  # Шаг оптимизатора + обновление LR
        
        train_loss += loss.item()  # Накопление loss
    
    # Validation
    model.eval()  # Режим валидации
    val_loss = 0
    with torch.no_grad():  # Без градиентов
        for i in range(0, len(val_src), batch_size):
            src = val_src[i:i+batch_size].to(device)
            tgt = val_tgt[i:i+batch_size].to(device)
            output = model(src)  # Прямой проход
            val_loss += criterion(output.reshape(-1, vocab_size), tgt.reshape(-1)).item()
    
    avg_train = train_loss / (len(train_src) / batch_size)
    avg_val = val_loss / (len(val_src) / batch_size)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    lr_history.append(scheduler.get_rate())
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")
    # Early stopping
    if avg_val < best_loss:  # Улучшили validation loss?
        best_loss = avg_val
        best_state = deepcopy(model.state_dict())  # Сохраняем лучшие веса
        wait = 0  # Счётчик эпох без улучшения
    else:
        wait += 1  # Увеличиваем счётчик
        if wait >= patience:  # Достигли лимита терпения?
            break  # Ранняя остановка

# Load best model
model.load_state_dict(best_state)  # Загружаем лучшие веса

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 4))  # Три графика в ряд

axes[0].plot(train_losses, label='Train')  # Train loss
axes[0].plot(val_losses, label='Val')  # Validation loss
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(lr_history)  # Скорость обучения
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning rate')
axes[1].set_title('LR Schedule')
axes[1].grid(True)

# Accuracy
model.eval()  # Режим валидации
with torch.no_grad():  # Без градиентов
    src_sample = val_src[:100].to(device)
    tgt_sample = val_tgt[:100].to(device)
    output = model(src_sample)
    preds = output.argmax(-1)  # Предсказанные токены
    non_pad = tgt_sample != PAD  # Маска: не-PAD токены
    acc = (preds[non_pad] == tgt_sample[non_pad]).float().mean().item()  # Точность на валидации

axes[2].bar(['Accuracy'], [acc])
axes[2].set_ylim(0, 1)
axes[2].set_title(f'Validation Accuracy: {acc:.2%}')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.show()

## 10.5 Сохранение и загрузка модели

In [ ]:

os.makedirs('models', exist_ok=True)  # Создаём директорию
checkpoint = {  # Сохраняем чекпоинт
    'epoch': epoch,  # Номер эпохи
    'model_state_dict': model.state_dict(),  # Веса модели
    'optimizer_state_dict': optimizer.state_dict(),  # Состояние оптимизатора
    'train_loss': train_losses[-1],
    'val_loss': val_losses[-1],
}
torch.save(checkpoint, 'models/transformer_checkpoint.pt')  # Сохраняем на диск
print("Model saved to models/transformer_checkpoint.pt")

# Загрузка
loaded = torch.load('models/transformer_checkpoint.pt', map_location=device)  # Загружаем чекпоинт
print(f"Loaded checkpoint: epoch={loaded['epoch']}, val_loss={loaded['val_loss']:.4f}")

## 10.6 Эксперименты: влияние гиперпараметров

Сравним разные конфигурации: d_model, learning rate, warmup.

In [ ]:

def train_quick(config):  # Быстрое обучение для сравнения
    m = Transformer(vocab_size=vocab_size, **config).to(device)  # Модель с заданной конфигурацией
    opt = torch.optim.AdamW(m.parameters(), lr=0.001, betas=(0.9, 0.98))
    sch = NoamScheduler(opt, d_model=config['d_model'], warmup_steps=30)  # Scheduler
    
    losses = []  # Лог потерь
    for _ in range(40):  # 40 эпох
        perm = torch.randperm(len(train_src))  # Перемешивание данных
        epoch_loss = 0
        for i in range(0, len(train_src), 64):
            idx = perm[i:i+64]
            src = train_src[idx].to(device)  # Исходные токены
            tgt = train_tgt[idx].to(device)  # Целевые токены
            out = m(src)
            loss = criterion(out.reshape(-1, vocab_size), tgt.reshape(-1))
            opt.zero_grad()
            loss.backward()  # Обратное распространение
            sch.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / (len(train_src) / 64))
    return losses[-1]

configs = [  # Список конфигураций
    {'d_model': 16, 'n_heads': 2, 'num_layers': 2, 'max_len': 12, 'dropout': 0.1},  # Маленькая
    {'d_model': 32, 'n_heads': 4, 'num_layers': 2, 'max_len': 12, 'dropout': 0.1},
    {'d_model': 64, 'n_heads': 4, 'num_layers': 3, 'max_len': 12, 'dropout': 0.1},  # Большая (wide)
    {'d_model': 32, 'n_heads': 4, 'num_layers': 4, 'max_len': 12, 'dropout': 0.1},  # Глубокая
]

results = []  # Результаты
for cfg in configs:  # Обучаем каждую конфигурацию
    loss = train_quick(cfg)
    label = f"d={cfg['d_model']} h={cfg['n_heads']} L={cfg['num_layers']}"  # Подпись для графика
    results.append((label, loss))  # Сохраняем результат

plt.figure(figsize=(10, 5))
labels, losses = zip(*results)
plt.bar(labels, losses)
plt.xlabel('Configuration')
plt.ylabel('Final loss')
plt.title('Hyperparameter Comparison')
plt.xticks(rotation=15)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Training Transformer complete ===")
print("Topics covered:")
print("  - CrossEntropyLoss with padding ignore")
print("  - AdamW optimizer")
print("  - Noam scheduler (warmup + decay)")
print("  - Full training loop with validation")
print("  - Early stopping")
print("  - Model checkpoint save/load")
print("  - Hyperparameter experiments")
print(f"  - Best validation loss: {best_loss:.4f}")